# Registering conditions onto the features DataFrame

This extends the `df` built in `example_workflow.ipynb` (one row per well,
one column per feature) with per-well experiment metadata from a layout
file — cell line, treatment, dose, batch, differentiation date, etc.

Real layout files (e.g. `layout/Bioni37A_3_050325.csv`) are one row per
well with explicit metadata columns, not the row/column-letter grid that
`MEA_features.read_layout` expects — so here the layout is just read with
pandas and merged onto `df` on `well`.

In [1]:
import os

import pandas as pd

from multiwell_mea_analysis.MEA_features import read_ephys, get_features, feature_names

## 1) Rebuild the per-well features DataFrame

Same as `example_workflow.ipynb` — see that notebook for the QC/raster steps.

In [2]:
file_path = os.path.join('data', 'CACNA1E_Bioni37A_29_3_baseline_20250305_20250403_spikes.csv')

spikes, bursts, netbursts = read_ephys(file_path)
wells = sorted(spikes['Well Label'].unique())

rows = [
    [file_path, well, *get_features(well, spikes, bursts, netbursts)]
    for well in wells
]
df = pd.DataFrame(rows, columns=['file_path', 'well', *feature_names])
df.shape

/Users/ovinogradov/Documents/Projects/MEA-analysis-base/multiwell_mea_analysis/MEA_features.py:120: RuntimeWarning: Mean of empty slice
  np.nanmean(mibis),
/Users/ovinogradov/Documents/Projects/MEA-analysis-base/multiwell_mea_analysis/MEA_features.py:121: RuntimeWarning: Mean of empty slice
  np.nanmean(cv_ibis),
/Users/ovinogradov/Documents/Projects/MEA-analysis-base/multiwell_mea_analysis/MEA_features.py:122: RuntimeWarning: Mean of empty slice
  np.nanmean(burst_rates),
/Users/ovinogradov/Documents/Projects/MEA-analysis-base/multiwell_mea_analysis/MEA_features.py:123: RuntimeWarning: Mean of empty slice
  np.nanmean(mbd),
/Users/ovinogradov/Documents/Projects/MEA-analysis-base/multiwell_mea_analysis/MEA_features.py:124: RuntimeWarning: Mean of empty slice
  np.nanmean(cv_bds),
/Users/ovinogradov/Documents/Projects/MEA-analysis-base/multiwell_mea_analysis/MEA_features.py:125: RuntimeWarning: Mean of empty slice
  np.nanmean(avg_rate_in_burst)
/Users/ovinogradov/miniconda3/envs/cacna

/Users/ovinogradov/miniconda3/envs/cacna1e_aso/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/ovinogradov/miniconda3/envs/cacna1e_aso/lib/python3.11/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


(24, 24)

## 2) Load the layout

`well` is the join key; `exclude` flags wells that shouldn't be analyzed;
`treatment`/`dose`/`dose_unit` describe the condition; `recording_name`
says which recording this layout row belongs to.

In [3]:
layout_path = os.path.join('layout', 'Bioni37A_3_050325.csv')
layout = pd.read_csv(layout_path)
layout[['well', 'cell_line', 'treatment', 'dose', 'dose_unit', 'batch', 'div', 'exclude']].head()

,well,cell_line,treatment,dose,dose_unit,batch,div,exclude
0,A1,Bioni37A,baseline,NaN,NaN,1,28,No
1,A2,Bioni37A,baseline,NaN,NaN,1,28,No
2,A3,Bioni37A,baseline,NaN,NaN,1,28,No
3,A4,Bioni37A,baseline,NaN,NaN,1,28,No
4,A5,Bioni37A,baseline,NaN,NaN,1,28,No


## 3) Sanity-check the layout matches this recording

Worth confirming before merging, especially once there are many (recording
file, layout file) pairs to keep straight.

In [4]:
recording_name = os.path.basename(file_path).rsplit('_spikes.csv', 1)[0]
assert (layout['recording_name'] == recording_name).all(), "layout doesn't match this recording"

## 4) Merge conditions onto df and drop excluded wells

In [5]:
def make_condition(row):
    """Human-readable condition label, e.g. 'baseline' or 'LTX5 1uM'."""
    if pd.isna(row['dose']):
        return row['treatment']
    return f"{row['treatment']} {row['dose']:g}{row['dose_unit']}"

layout = layout.copy()
layout['condition'] = layout.apply(make_condition, axis=1)

condition_cols = ['well', 'cell_line', 'condition', 'treatment', 'dose', 'dose_unit',
                  'batch', 'div', 'differentiation_date', 'recording_date', 'exclude']
df = df.merge(layout[condition_cols], on='well', how='left')

# Layout-flagged exclusions shouldn't be analyzed
df = df[df['exclude'] != 'Yes'].drop(columns='exclude').reset_index(drop=True)

df[['well', 'condition', 'cell_line', 'batch', 'div', 'ISI [s]', 'FR [Hz]']].head()

,well,condition,cell_line,batch,div,ISI [s],FR [Hz]
0,A1,baseline,Bioni37A,1,28,0.331929,4.980823
1,A2,baseline,Bioni37A,1,28,0.184327,6.083098
2,A3,baseline,Bioni37A,1,28,0.288130,4.351575
3,A4,baseline,Bioni37A,1,28,0.124922,8.505921
4,A5,baseline,Bioni37A,1,28,1.290092,6.676835


For many recording files at once, the per-file processors in
`MEA_features.py` (`process_recording_file`, `process_multidrug_recording_file`,
`process_plate_layout_recording_file`, ...) do this same per-file layout
merge internally and return one row per well, ready to `pd.concat` across
files into a single conditions table — see their docstrings for which
layout convention each one expects.